# Stage 2 — Your first neural network (MLP in PyTorch)

**Goal:** write a training loop yourself, watch a network learn and overfit, and find out whether it beats the Stage 1 baseline
(linear SVM: **46.8 ± 3.1 %** top-1 in 5-fold cross-validation, 300 signs, chance 0.33 %).

**What you will learn here**
| Concept | Where |
|---|---|
| Training loop: forward pass, loss, backward pass, optimizer step | Cell 7 (`fit`) |
| "Can the model memorize 64 clips?" sanity check | Cell 9 |
| Learning curves and **overfitting** | Cells 10–11 |
| Dropout, weight decay, label smoothing, early stopping | Cell 11 |
| Hand-made features vs the raw frame sequence | Cells 10 and 12 |
| Seeds and run-to-run variation | Cell 13 |
| Honest evaluation: same 5 folds as Stage 1 | Cell 14 |
| Does a bigger model exploit signer identity (leakage)? | Cell 15 |

**Two kinds of input**
- `features`: the 648 summary numbers from Stage 1 (needs `features_stage1_subset_T64_S3.npz` in your Drive).
- `sequence`: the clip itself, 64 frames x 132 numbers per frame (normalized hands, wrist position, hand visible), flattened.

**How to run:** a **GPU runtime is recommended** (Runtime > Change runtime type > T4 GPU), but the notebook also works on CPU.
On CPU, set `RUN_SEQUENCE_MLP = False` in Cell 2 for a first pass (the sequence network is about 10x slower).
Rough times (estimates, not measured): features network 15–60 s per run; sequence network 20–60 s per run on GPU, 3–8 min on CPU.

In [ ]:
# [setup] Cell 2 — configuration
from pathlib import Path
import json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
FIXED_T = 64              # frames per clip
N_FOLDS = 5
RUN_SEQUENCE_MLP = True   # the network that reads the raw frame sequence (slow on CPU)
SEEDS = [0, 1, 2]         # run-to-run variation
warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'no GPU')

In [ ]:
# Cell 3 — load the Stage 0 outputs (same files and same split as Stage 1)
splits = pd.read_csv(PROC / 'splits.csv')
z = np.load(PROC / 'landmarks_raw.npz', allow_pickle=False)
ids, lengths = z['ids'], z['lengths']
offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
frames = np.load(PROC / 'frames_tracked_v2.npy', mmap_mode='r')
assert len(frames) == offsets[-1], 'frames_tracked_v2.npy does not match landmarks_raw.npz: re-run Stage 0 cells 10 and 11b.'

meta = splits[splits['in_subset']].reset_index(drop=True)              # development subset (same clips and order as Stage 1)
y_all = meta['label_id_subset'].astype(int).to_numpy()
N_CLASSES = int(y_all.max()) + 1
assert len(np.unique(y_all)) == N_CLASSES

tr = (meta['split'] == 'train').to_numpy(); va = (meta['split'] == 'val').to_numpy(); te = (meta['split'] == 'test').to_numpy()
folds = meta['fold'].to_numpy()                                         # -1 = prolific signers (always train)
print(f'{len(meta)} clips | {N_CLASSES} signs | chance {1 / N_CLASSES:.2%} | train {tr.sum()} val {va.sum()} test {te.sum()}')

In [ ]:
# [sequence] Cell 4 — the raw sequence input: 64 frames x 132 numbers per clip (cached)
from tqdm.auto import tqdm
WRIST, MIDDLE_MCP = 0, 9

def normalize_hands(x, aspect=None, eps=1e-6):
    """x: (..., 21, 3). Aspect-correct x, centre on the wrist, divide by the wrist->middle-finger-base distance."""
    x = x.astype(np.float32).copy()
    if aspect is not None:
        x[..., 0] *= aspect
    centred = x - x[..., WRIST:WRIST + 1, :]
    scale = np.linalg.norm(centred[..., MIDDLE_MCP, :2], axis=-1)
    scale = np.where(scale < eps, np.nan, scale)
    return centred / scale[..., None, None]

def clip_to_sequence(clip, aspect, T=FIXED_T):
    """(L, 2, 21, 3) -> (T, 132): 126 normalized joint values, 4 wrist-position values, 2 'hand visible' flags.
    Missing hands are filled with 0 (and flagged by the visibility channels)."""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    hands = normalize_hands(clip, aspect=aspect)[idx]                       # (T, 2, 21, 3)
    wrist = clip[:, :, WRIST, :2].astype(np.float32).copy()
    wrist[..., 0] *= aspect
    wrist = wrist[idx]                                                      # (T, 2, 2)
    present = ~np.isnan(hands).all(axis=(2, 3))                             # (T, 2)
    hands = np.where(np.isnan(hands), 0.0, hands)
    wrist = np.where(present[:, :, None], wrist, 0.0)
    return np.concatenate([hands.reshape(T, 126), wrist.reshape(T, 4), present.astype(np.float32)], axis=1).astype(np.float32)

# --- unit checks ---
_rng = np.random.default_rng(0)
_clip = _rng.normal(size=(40, 2, 21, 3)).astype(np.float32); _clip[10:15, 1] = np.nan       # hand 1 missing in 5 frames
_s = clip_to_sequence(_clip, 1.0)
assert _s.shape == (FIXED_T, 132) and np.isfinite(_s).all()
assert _s[:, 131].min() == 0.0 and _s[:, 130].min() == 1.0              # slot 1 flagged missing somewhere, slot 0 always visible
_moved = clip_to_sequence(_clip * 2.0 + np.array([5.0, -3.0, 0.0], np.float32), 1.0)
assert np.allclose(_s[:, :126], _moved[:, :126], atol=1e-4), 'joint channels must not depend on hand position or size'
print('OK: sequence unit checks passed')

SEQF = PROC / f'sequences_subset_T{FIXED_T}.npz'
meta_ids = meta['attachment_id'].to_numpy().astype(str)
SEQ = None
if SEQF.exists():
    zs = np.load(SEQF, allow_pickle=False)
    if zs['ids'].shape == meta_ids.shape and (zs['ids'] == meta_ids).all():
        SEQ = zs['S'].astype(np.float32)
        print('Loaded cached sequences:', SEQF.name)
if SEQ is None:
    SEQ = np.zeros((len(meta), FIXED_T, 132), np.float32)
    for r, (cid, asp) in enumerate(tqdm(zip(meta['attachment_id'], meta['aspect']), total=len(meta))):
        i = id2idx[cid]
        SEQ[r] = clip_to_sequence(np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32), float(asp))
    np.savez_compressed(SEQF, S=SEQ.astype(np.float16), ids=meta_ids)
    print('Saved', SEQF.name)
print('sequence tensor:', SEQ.shape, '| flattened input size:', SEQ.shape[1] * SEQ.shape[2])

In [ ]:
# Cell 5 — the Stage 1 summary features (648 numbers per clip)
FEATF = PROC / f'features_stage1_subset_T{FIXED_T}_S3.npz'
assert FEATF.exists(), f'{FEATF.name} not found in {PROC}: run Stage 1 cell 4 first (it creates this file).'
zf = np.load(FEATF, allow_pickle=False)
assert (zf['ids'] == meta_ids).all(), 'feature rows are in a different order than the Stage 0 split: re-run Stage 1 cell 4.'
FEATS = zf['X'].astype(np.float32)
print('summary features:', FEATS.shape)

In [ ]:
# [prep] Cell 6 — normalization from TRAIN rows only, then the arrays the network sees
def seq_mask(S):
    """(N, T, 132) mask: 1 where the channel's hand is visible (or the channel is a visibility flag)."""
    p = S[:, :, 130:132]
    return np.concatenate([np.repeat(p[:, :, :1], 63, 2), np.repeat(p[:, :, 1:], 63, 2),
                           np.repeat(p[:, :, :1], 2, 2), np.repeat(p[:, :, 1:], 2, 2), np.ones_like(p)], axis=2)

def seq_stats(S_tr):
    m = seq_mask(S_tr)
    cnt = np.maximum(m.sum((0, 1)), 1.0)
    mean = (S_tr * m).sum((0, 1)) / cnt
    std = np.sqrt((((S_tr - mean) ** 2) * m).sum((0, 1)) / cnt)
    std = np.where(std < 1e-3, 1.0, std)                                   # a channel that never varies in train: leave it alone
    mean[130:], std[130:] = 0.0, 1.0                                        # leave the visibility flags as 0/1
    return mean.astype(np.float32), std.astype(np.float32)

def apply_seq_norm(S, mean, std):
    return (((S - mean) / std) * seq_mask(S)).astype(np.float32)           # missing hands stay exactly 0

def prepare(kind, train_mask, masks):
    """kind: 'features' or 'sequence'. Mean/std come from the TRAIN rows only (no leakage). Returns {name: array}."""
    if kind == 'features':
        mean = FEATS[train_mask].mean(0); std = FEATS[train_mask].std(0)
        std = np.where(std < 1e-6, 1.0, std)                               # constant features: leave them alone (as StandardScaler does)
        return {k: ((FEATS[m] - mean) / std).astype(np.float32) for k, m in masks.items()}
    mean, std = seq_stats(SEQ[train_mask])
    return {k: apply_seq_norm(SEQ[m], mean, std).reshape(int(m.sum()), -1) for k, m in masks.items()}

# --- unit checks ---
_d = prepare('features', tr, {'train': tr, 'val': va})
assert _d['train'].shape == (tr.sum(), FEATS.shape[1]) and np.isfinite(_d['train']).all()
_q = prepare('sequence', tr, {'train': tr})['train']
assert _q.shape == (tr.sum(), FIXED_T * 132) and np.isfinite(_q).all()
print('OK: prepared inputs -> features', _d['train'].shape, '| sequence', _q.shape)
del _d, _q

In [ ]:
# [torch] Cell 7 — the neural network and the training loop (this is the heart of Stage 2)
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class MLP(nn.Module):
    """Linear -> ReLU -> Dropout, repeated, then a final Linear layer that outputs one score (logit) per sign."""
    def __init__(self, in_dim, n_classes, hidden=(512, 256), dropout=0.3):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def new_model(in_dim, hidden, dropout, seed):
    set_seed(seed)
    return MLP(in_dim, N_CLASSES, hidden, dropout).to(DEVICE)

@torch.no_grad()
def _eval(model, X, y):
    model.eval()                                   # dropout off
    logits = model(X)
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, Xtr, ytr, Xva=None, yva=None, epochs=150, lr=1e-3, weight_decay=1e-2, batch_size=128,
        patience=20, label_smoothing=0.0, seed=0, fixed_epochs=None):
    """Train with mini-batches. With a validation set: stop when validation accuracy has not improved for `patience`
    epochs and restore the best weights. With fixed_epochs: train exactly that many epochs, no validation."""
    t0 = time.time()
    n_epochs = fixed_epochs or epochs
    g = torch.Generator(); g.manual_seed(seed)                         # controls the order of the mini-batches
    Xt = torch.as_tensor(Xtr, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    use_val = (Xva is not None) and (fixed_epochs is None)
    if use_val:
        Xv = torch.as_tensor(Xva, dtype=torch.float32, device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_epoch, best_state, bad, n = -1.0, 0, None, 0, len(Xt)
    for epoch in range(1, n_epochs + 1):
        model.train()                                                  # dropout on
        perm = torch.randperm(n, generator=g).to(DEVICE)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])                   # forward pass + loss
            loss.backward()                                            # backward pass: gradients
            opt.step()                                                 # update the weights
        tl, ta = _eval(model, Xt, yt)
        hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        if use_val:
            vl, vacc = _eval(model, Xv, yv)
            hist['val_loss'].append(vl); hist['val_acc'].append(vacc)
            if vacc > best_acc:
                best_acc, best_epoch, bad = vacc, epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_val and best_state is not None:
        model.load_state_dict(best_state)                              # go back to the best epoch
    else:
        best_epoch = n_epochs
    return hist, best_epoch, time.time() - t0

@torch.no_grad()
def predict_logits(model, X, batch_size=1024):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[i:i + batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

print('model and training loop defined')

In [ ]:
# [runner] Cell 8 — metrics and the experiment runner
from sklearn.metrics import f1_score

def metrics_from_logits(logits, y):
    pred = logits.argmax(1)
    top5 = np.argsort(-logits, axis=1)[:, :5]
    return {'n': int(len(y)), 'top1': float((pred == y).mean()), 'top5': float(np.mean([yt in t for yt, t in zip(y, top5)])),
            'macro_f1': float(f1_score(y, pred, average='macro')), 'pred': pred}

DEFAULT_CFG = dict(hidden=(512, 256), dropout=0.3, lr=1e-3, weight_decay=1e-2, batch_size=128,
                   epochs=150, patience=20, label_smoothing=0.0)
results = []

def run_one(experiment, name, kind, masks, seed, cfg=None, data=None, fixed_epochs=None):
    """masks must contain 'train' and (unless fixed_epochs) 'val'; every other entry is just evaluated."""
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    if data is None:
        data = prepare(kind, masks['train'], masks)
    model = new_model(data['train'].shape[1], cfg['hidden'], cfg['dropout'], seed)
    hist, best_ep, fit_s = fit(model, data['train'], y_all[masks['train']],
                               data.get('val'), y_all[masks['val']] if 'val' in masks else None,
                               epochs=cfg['epochs'], lr=cfg['lr'], weight_decay=cfg['weight_decay'],
                               batch_size=cfg['batch_size'], patience=cfg['patience'],
                               label_smoothing=cfg['label_smoothing'], seed=seed, fixed_epochs=fixed_epochs)
    out = {'name': name, 'kind': kind, 'seed': seed, 'hist': hist, 'best_epoch': best_ep, 'fit_s': fit_s, 'metrics': {}}
    for sname, m in masks.items():
        if sname == 'train':
            continue
        r = metrics_from_logits(predict_logits(model, data[sname]), y_all[m])
        out['metrics'][sname] = r
        results.append({'experiment': experiment, 'name': name, 'kind': kind, 'seed': seed, 'set': sname, 'n': r['n'],
                        'top1': r['top1'], 'top5': r['top5'], 'macro_f1': r['macro_f1'],
                        'best_epoch': best_ep, 'fit_s': fit_s})
    return out

def summary_line(o, sets=('val', 'test')):
    return ' | '.join(f'{s} top-1 {o["metrics"][s]["top1"]:.1%} top-5 {o["metrics"][s]["top5"]:.1%}' for s in sets if s in o['metrics']) \
           + f' | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s'

MASKS = {'train': tr, 'val': va, 'test': te}

In [ ]:
# Cell 9 — sanity check: can the network memorize 64 training clips? (if not, the training loop has a bug)
d = prepare('features', tr, {'train': tr})
small = np.random.default_rng(0).choice(np.where(tr)[0], 64, replace=False)
sel = np.isin(np.where(tr)[0], small)
mdl = new_model(d['train'].shape[1], (512, 256), 0.0, 0)
h, _, _ = fit(mdl, d['train'][sel], y_all[tr][sel], fixed_epochs=300, lr=3e-3, weight_decay=0.0, batch_size=64, seed=0)
print(f'train loss {h["train_loss"][0]:.2f} -> {h["train_loss"][-1]:.3f} | train accuracy {h["train_acc"][0]:.0%} -> {h["train_acc"][-1]:.0%}')
assert h['train_acc'][-1] > 0.9, 'The network cannot even memorize 64 clips: something is wrong with the training loop.'
print('OK: the network can learn (it memorizes the 64 clips)')
del d, mdl

In [ ]:
# Cell 10 — Experiment 1: your first MLP on the 648 summary features, and its learning curves
DATA_F = prepare('features', tr, MASKS)                                  # normalize once, reuse
o1 = run_one('E1 first MLP', 'MLP on features', 'features', MASKS, seed=0, data=DATA_F)
print(summary_line(o1))

h = o1['hist']; ep = np.arange(1, len(h['train_loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ep, h['train_loss'], label='train'); ax[0].plot(ep, h['val_loss'], label='validation')
ax[0].axvline(o1['best_epoch'], color='k', ls=':', label='best epoch'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title('Loss')
ax[1].plot(ep, h['train_acc'], label='train'); ax[1].plot(ep, h['val_acc'], label='validation')
ax[1].axhline(0.468, color='gray', ls='--', label='Stage 1 SVM (CV)'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].set_title('Accuracy')
plt.tight_layout(); plt.savefig(FIG / 'stage2_e1_curves.png', dpi=150); plt.show()

In [ ]:
# Cell 11 — Experiment 2: overfitting and how to fight it (regularization)
E2_CONFIGS = [('no regularization', dict(dropout=0.0, weight_decay=0.0)),
              ('dropout 0.3 + weight decay 0.01', {}),
              ('dropout 0.5 + weight decay 0.05', dict(dropout=0.5, weight_decay=0.05)),
              ('dropout 0.3 + label smoothing 0.1', dict(label_smoothing=0.1))]
e2 = []
for name, cfg in E2_CONFIGS:
    o = run_one('E2 regularization', name, 'features', MASKS, seed=0, cfg=cfg, data=DATA_F)
    e2.append((name, cfg, o))
    print(f'{name:36s} {summary_line(o)} | final train acc {o["hist"]["train_acc"][-1]:.0%}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, cfg, o in e2:
    ep = np.arange(1, len(o['hist']['val_acc']) + 1)
    ax[0].plot(ep, o['hist']['val_acc'], label=name); ax[1].plot(ep, o['hist']['train_acc'], label=name)
ax[0].set_title('Validation accuracy (unseen signers)'); ax[1].set_title('Training accuracy'); ax[0].legend(fontsize=7)
for a in ax:
    a.set_xlabel('epoch')
plt.tight_layout(); plt.savefig(FIG / 'stage2_e2_regularization.png', dpi=150); plt.show()

best_e2 = max(e2, key=lambda t: t[2]['metrics']['val']['top1'])         # chosen on VALIDATION, never on test
BEST_CFG = {'features': best_e2[1]}
print('chosen for the features network:', best_e2[0])

In [ ]:
# Cell 12 — Experiment 3: an MLP that reads the raw frame sequence (64 frames x 132 numbers, flattened)
DATA_S = None
if RUN_SEQUENCE_MLP:
    DATA_S = prepare('sequence', tr, MASKS)
    cfg_s = dict(dropout=0.5, weight_decay=0.05)
    o3 = run_one('E3 sequence MLP', 'MLP on sequence', 'sequence', MASKS, seed=0, cfg=cfg_s, data=DATA_S)
    BEST_CFG['sequence'] = cfg_s
    print(summary_line(o3), f'| final train acc {o3["hist"]["train_acc"][-1]:.0%}')
    ep = np.arange(1, len(o3['hist']['val_acc']) + 1)
    plt.figure(figsize=(5.5, 3.6)); plt.plot(ep, o3['hist']['train_acc'], label='train'); plt.plot(ep, o3['hist']['val_acc'], label='validation')
    plt.axhline(0.468, color='gray', ls='--', label='Stage 1 SVM (CV)'); plt.xlabel('epoch'); plt.legend(); plt.title('Sequence MLP')
    plt.tight_layout(); plt.savefig(FIG / 'stage2_e3_sequence.png', dpi=150); plt.show()
else:
    print('RUN_SEQUENCE_MLP is False: skipped')

In [ ]:
# Cell 13 — Experiment 4: run-to-run variation (3 seeds per network)
final_runs = {}
for kind, data in (('features', DATA_F), ('sequence', DATA_S)):
    if data is None:
        continue
    final_runs[kind] = [run_one('E4 seeds', f'MLP on {kind}', kind, MASKS, seed=s, cfg=BEST_CFG[kind], data=data) for s in SEEDS]
    v = [o['metrics']['val']['top1'] for o in final_runs[kind]]; t = [o['metrics']['test']['top1'] for o in final_runs[kind]]
    t5 = [o['metrics']['test']['top5'] for o in final_runs[kind]]
    print(f'{kind:9s} val top-1 {np.mean(v):.1%} +/- {np.std(v):.1%} | test top-1 {np.mean(t):.1%} +/- {np.std(t):.1%} '
          f'| test top-5 {np.mean(t5):.1%} | per-seed test: ' + ', '.join(f'{x:.1%}' for x in t))

BEST_KIND = max(final_runs, key=lambda k: np.mean([o['metrics']['val']['top1'] for o in final_runs[k]]))
BEST_EPOCH = int(np.median([o['best_epoch'] for o in final_runs[BEST_KIND]]))
print(f'\nbest input type on validation: {BEST_KIND} (median best epoch {BEST_EPOCH})')

In [ ]:
# Cell 14 — Experiment 5: 5-fold cross-validation with the best network (same folds as Stage 1)
cv_acc, pooled_y, pooled_p = [], [], []
for k in range(N_FOLDS):
    te_k = folds == k; va_k = folds == (k + 1) % N_FOLDS; tr_k = ~(te_k | va_k)       # prolific signers (fold -1) stay in train
    o = run_one('E5 5-fold CV', f'MLP on {BEST_KIND}', BEST_KIND, {'train': tr_k, 'val': va_k, 'test': te_k},
                seed=0, cfg=BEST_CFG[BEST_KIND])
    results[-2]['set'], results[-1]['set'] = f'fold {k} val', f'fold {k}'                # label the rows of this fold
    cv_acc.append(o['metrics']['test']['top1']); pooled_y.append(y_all[te_k]); pooled_p.append(o['metrics']['test']['pred'])
    print(f'fold {k}: test clips {te_k.sum()} | top-1 {cv_acc[-1]:.1%} | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s')
pooled_y, pooled_p = np.concatenate(pooled_y), np.concatenate(pooled_p)
print(f'\nMLP on {BEST_KIND}: top-1 mean {np.mean(cv_acc):.1%} +/- {np.std(cv_acc):.1%} over folds | pooled {np.mean(pooled_y == pooled_p):.1%} on {len(pooled_y)} clips')

s1_path = RES / 'stage1_results.csv'
if s1_path.exists():
    s1 = pd.read_csv(s1_path)
    s1_cv = s1[(s1.experiment == '5-fold CV')]
    for mname in s1_cv.model.unique():
        v = s1_cv[s1_cv.model == mname].sort_values('set').top1.to_numpy()
        if len(v) == N_FOLDS:
            print(f'  Stage 1 {mname:28s}: {v.mean():.1%} +/- {v.std():.1%} | MLP better in {(np.array(cv_acc) > v).sum()} of {N_FOLDS} folds')
else:
    print('(results/stage1_results.csv not found: Stage 1 numbers are 31.2%, 43.0% and 46.8% for kNN, logistic regression and SVM)')

In [ ]:
# Cell 15 — Experiment 6: does a bigger model exploit signer identity? (official split: seen vs unseen signers)
off_train = meta['official_train'].to_numpy(bool)
unseen = meta['official_test_unseen_signer'].to_numpy(bool)
seen = (~off_train) & (~unseen)
print(f'official train {off_train.sum()} | official test: {seen.sum()} clips from seen signers, {unseen.sum()} from unseen signers | fixed epochs {BEST_EPOCH}')
leak = [run_one('E6 leakage', f'MLP on {BEST_KIND}', BEST_KIND, {'train': off_train, 'seen': seen, 'unseen': unseen},
                seed=s, cfg=BEST_CFG[BEST_KIND], fixed_epochs=BEST_EPOCH) for s in SEEDS]
a = [o['metrics']['seen']['top1'] for o in leak]; b = [o['metrics']['unseen']['top1'] for o in leak]
print(f'seen signers {np.mean(a):.1%} +/- {np.std(a):.1%} | unseen signers {np.mean(b):.1%} +/- {np.std(b):.1%} | gap {np.mean(a) - np.mean(b):+.1%}')
print('(Stage 1, logistic regression: seen 38.4%, unseen 37.8%)')

In [ ]:
# Cell 16 — results table and the final figure
res = pd.DataFrame(results)
res.to_csv(RES / 'stage2_results.csv', index=False)
print(res.groupby(['experiment', 'name', 'set'])[['top1', 'top5', 'macro_f1']].mean().round(3))

labels, means, stds = [], [], []
if s1_path.exists():
    for mname in ['kNN', 'PCA + logistic regression', 'linear SVM']:
        v = s1_cv[s1_cv.model == mname].top1.to_numpy()
        if len(v) == N_FOLDS:
            labels.append('Stage 1: ' + mname); means.append(v.mean()); stds.append(v.std())
labels.append(f'Stage 2: MLP on {BEST_KIND}'); means.append(np.mean(cv_acc)); stds.append(np.std(cv_acc))
plt.figure(figsize=(8, 3.8))
plt.bar(range(len(labels)), means, yerr=stds, capsize=4, color=['#999'] * (len(labels) - 1) + ['#4C78A8'])
plt.xticks(range(len(labels)), labels, rotation=20, ha='right'); plt.ylabel('top-1 accuracy (5-fold CV, mean +/- std)')
plt.axhline(1 / N_CLASSES, color='k', ls='--', lw=1); plt.title('Stage 1 baselines vs the first neural network')
plt.tight_layout(); plt.savefig(FIG / 'stage2_results.png', dpi=150); plt.show()
print('Saved', RES / 'stage2_results.csv', 'and', FIG / 'stage2_results.png')

## Stage 2 checklist

- [ ] Cell 9 printed `OK: the network can learn`
- [ ] Cell 10: curves show train accuracy rising above validation accuracy (overfitting) and the best epoch marked
- [ ] Cell 11: you can see which regularization setting gave the best validation accuracy
- [ ] Cell 13: results for 3 seeds (mean +/- std)
- [ ] Cell 14: 5-fold CV for the best network, compared with the Stage 1 SVM
- [ ] `results/stage2_results.csv` and the figures exist

**Send back:** the printed outputs of Cells 9, 10, 11, 12, 13, 14 and 15, plus the figures `stage2_e1_curves.png`,
`stage2_e2_regularization.png` and `stage2_results.png`. If a cell fails, send me the last lines of the error.